# Mathematical Analysis of a Combined Filter
Last update: 25 Mar 2026

See the link: [mathematical analysis of a combined filter](https://electronics.stackexchange.com/questions/766702/mathematical-analysis-of-a-combined-filter)

---

The structure shown below in the photo and video has common mode choke which creates common mode filter and the leakage inductor is the diffirential mode filter. Inductor impedance is ZL=jωL, capacitor impedance ZC=1/jωC." 

How do I derive mathematically the differential mode filter transfer function?

How do I derive mathematically the common mode filter transfer function?

https://www.youtube.com/watch?v=8M8B8GytW78

![schematic](cpqHz5gY.png)

asked Mar 8 at 13:35  
rocko445  
43511 gold badge33 silver badges11  

edited Mar 8 at 14:01  
greybeard  
3,72633 gold badges1111 silver badges26  

---

### Comments

>Indicate input and output ports. –  
Andy aka  
Commented Mar 8 at 13:42  

>This one has separate inductors for differential mode filtering. Of course, the leakage inductances of part L_CM1 also affect differential mode noises. To get the transfer function you should specify source impedance, load impedance, resistances of inductors an the coupling factor of the inductors in L_CM1. All these can be, of course, symbols. I'm afraid the general transfer function formula is too massive to be useful and deriving it is so big effort that only the owners of special symbolic circuit analysis software can get it right An opinion: Better to try numerical plots. –   
oneprivate  
Commented Mar 8 at 18:08  

>is there a practical example I could simulate? –   
rocko445  
Commented Mar 10 at 14:37   

---

## My Answer

The circuit in the schematic is of a type of flter used to reduce conducted emmisions from a [Switch Mode Power Supply](https://en.wikipedia.org/wiki/Switched-mode_power_supply) (SMPS). The schematic was presented in a video series that provides a guidance for designing Electromagnetic Compatibility (EMC) filters. I have redrawn the schematic using LTSpice, assigned node numbers and given referensece designators to the componets using numerical subscrips. The use of numberical subscrips will keep the symbolic equations from being too long when displayed. In the schemtic below, $V1$ is the DC source of power for the SMPS. The SMPS is connected to the filter at circuit nodes 4 and 6. 

![Schematic](mathematical_analysis_of_a_combined_filter_v1.png)

In the context of the question being asked, which concerns conducted emissions, it is the switching noise from the SMPS that filter is designed to attenuate. DC power flows from left to right in the schematic and the signal of interest (the noise) flows (or conducts) from the SMPS to the rest of the system from right to left through the filter. In other words, the filter's driving function is on the right side of the schematic as shown below.

At node 8 there is a connection to the chassis. In the example the term _chassis_ is used interchangeably with the term _protective earth_. In the analysis that follows, the chassis is considered a circuit node. No assumtions are made about connections between the chassis and other grounds in the system. 

@rocko445 asked the following questions: 

>How do I derive mathematically the differential mode filter transfer function?  

>How do I derive mathematically the common mode filter transfer function?  

The filter transfer function for the two cases can be obtained by applying modified nodal analysis to the circuit. My proposed solution was developed using Python in a JupyterLab notebook which is available on GitHub here]() (this GitHub repo is owned by Tiburonboy, the author of this answer). What you see below are parts of the notebook pasted into the answer dialog box.

The following Python modules are used in the notebook.

In [1]:
from sympy import *
import numpy as np
import pandas as pd
import SymMNA
from IPython.display import display, Markdown, Math, Latex
init_printing()

## Differntial Mode Case
The schematic shown in the question was redrawn using LTSpice and is shown below. The source $V_{dm}$ represents the differntial mode noise from the SMPS. According to the video, $L_1$ is the inductance from the DC supply. $R_2$ was added to the schematic to represent the source restance of $V_1$. For the differentical mode case, $V_1$ is set to zero and removed from the circuit. For this analysis, the filter's out put is node 2 in the schematic below. 

![Schematic](mathematical_analysis_of_a_combined_filter_v1_DM.png)

The net list for the schematic was copied from LTSpice and pasted into the code cell below. All the componet values have been set to 1. The coupling constant between $L_2$ and $L_3$ is also set to 1. Since we are intrested in the symbolic solution the componet values are not imporrtant. 

In [2]:
net_list_dm = '''
L1 1 2 1
L2 2 3 1
L3 0 7 1
L4 3 4 1
L5 6 7 1
C1 2 0 1
C2 7 8 1
C3 3 8 1
C4 4 6 1
C5 4 5 1
R1 5 6 1
V_dm 4 6 1
R2 1 0 1
K L2 L3 1
'''

The [Modified Nodal Analysis](https://en.wikipedia.org/wiki/Modified_nodal_analysis) (MNA) procedure was used to generate the network equations from the schematic's netlist. The procedure is implemented in Python, and the function `SymMNA.smna(net_list)` returns the network matrices along with a report and two Pandas data frames. The Python module available on GitHub at [SymMNA](https://github.com/Tiburonboy/SymMNA) (this GitHub repo is owned by Tiburonboy, the author of this answer). Several parameters are returned by the function, but only the matrices $A$, $X$ and $Z$ are used in the analysis below.

In [3]:
report, network_df, i_unk_df, A, X, Z = SymMNA.smna(net_list_dm)

The code below assembles the network equations from the MNA matrices and displays the equations.

In [4]:
# Put matrices into SymPy 
X = Matrix(X)
Z = Matrix(Z)

# formulate the symbolic network equations
NE_sym = Eq(A*X,Z)

# the free symbols are entered as SymPy variables and the element values are put into a dictionary.
var(str(NE_sym.free_symbols).replace('{','').replace('}',''))

# display the network equations
temp = ''
for i in range(shape(NE_sym.lhs)[0]):
    temp += '${:s} = {:s}$<br>'.format(latex(NE_sym.rhs[i]),latex(NE_sym.lhs[i]))
Markdown(temp)

$0 = I_{L1} + \frac{v_{1}}{R_{2}}$<br>$0 = C_{1} s v_{2} - I_{L1} + I_{L2}$<br>$0 = C_{3} s v_{3} - C_{3} s v_{8} - I_{L2} + I_{L4}$<br>$0 = - C_{4} s v_{6} - C_{5} s v_{5} - I_{L4} + I_{V dm} + v_{4} \left(C_{4} s + C_{5} s\right)$<br>$0 = - C_{5} s v_{4} + v_{5} \left(C_{5} s + \frac{1}{R_{1}}\right) - \frac{v_{6}}{R_{1}}$<br>$0 = - C_{4} s v_{4} + I_{L5} - I_{V dm} + v_{6} \left(C_{4} s + \frac{1}{R_{1}}\right) - \frac{v_{5}}{R_{1}}$<br>$0 = C_{2} s v_{7} - C_{2} s v_{8} - I_{L3} - I_{L5}$<br>$0 = - C_{2} s v_{7} - C_{3} s v_{3} + v_{8} \left(C_{2} s + C_{3} s\right)$<br>$V_{dm} = v_{4} - v_{6}$<br>$0 = - I_{L1} L_{1} s + v_{1} - v_{2}$<br>$0 = - I_{L2} L_{2} s - I_{L3} M s + v_{2} - v_{3}$<br>$0 = - I_{L2} M s - I_{L3} L_{3} s - v_{7}$<br>$0 = - I_{L4} L_{4} s + v_{3} - v_{4}$<br>$0 = - I_{L5} L_{5} s + v_{6} - v_{7}$<br>

In the equations above, $M$ is the mutual inductance and $M = K\sqrt{L_2L_3}$. 

The network equations for the netlist can be solved symbolically by using the SymPy fumction, `solve`. The time to get the solution is measured by IPython's magic command `%%time`. 

In [5]:
%%time
U_sym_dm = solve(NE_sym,X)

CPU times: user 1.26 s, sys: 7.57 ms, total: 1.27 s
Wall time: 1.27 s


The symbolic solution to the network equations took a bit over 1 second on my i3 CPU. The symbolic solution for the unknown currents and node voltages are long and not displayed in this answer.

The transfer function for the differntial mode case is: $H(s)=\frac {v_2(s)}{v_4(s)-v_6(s)}$. 

In [6]:
H_sym_dm = (U_sym_dm[v2]/(U_sym_dm[v4] - U_sym_dm[v6])).simplify().collect(s)

The expression for the transfer function can be simplified by letting $L_2=L_3=L$, $C_2=C_3=C$, $L_5=L_4$ and $M=1$. 

In [7]:
L, C = symbols('L C')
H_sym_dm_simplified = H_sym_dm.subs({L2:L,L3:L,L5:L4,C2:C,C3:C,M:1}).simplify().collect(s)

Markdown('$H(s)={:s}$'.format(latex(H_sym_dm_simplified)))

$H(s)=\frac{L_{1} s + R_{2}}{2 C C_{1} L_{1} L_{4} s^{5} \left(L - 1\right) + 2 C C_{1} L_{4} R_{2} s^{4} \left(L - 1\right) + R_{2} s^{2} \left(C L_{4} + 2 C_{1} L + 2 C_{1} L_{4} - 2 C_{1}\right) + R_{2} + s^{3} \cdot \left(2 C L L_{4} + C L_{1} L_{4} - 2 C L_{4} + 2 C_{1} L L_{1} + 2 C_{1} L_{1} L_{4} - 2 C_{1} L_{1}\right) + s \left(2 L + L_{1} + 2 L_{4} - 2\right)}$

Hopefully this answers your first question, "How do I derive mathematically the differential mode filter transfer function?"

## Common Mode Case
The circuit for the common mode case is shown below. The common mode voltage source, $V_{cm}$ is connected to both nodes 4 and 6 on one end and to the chassis on the other end. Since nodes 4 and 6 have been tied together, $R_1$, $C_4$ and $C5$ are shorted out and have been removed from the circuit. The nodes have been renumbered, because my Python code needs the nodes to be consectulive numbered. 

![Schematic](mathematical_analysis_of_a_combined_filter_v1_CM.png)

The netlist for the schematic was copied into the code cell below.

In [8]:
net_list_cm = '''
L1 1 2 1
L2 2 3 1
L3 0 5 1
L4 3 4 1
L5 4 5 1
C1 2 0 1
C2 5 6 1
C3 3 6 1
V_cm 4 6 1
R2 1 0 1
K L2 L3 1
'''

Call the symbolic modified nodal analysis function, `SymMNA.smna(net_list)` to generate the MNA network equations.

In [9]:
report, network_df, i_unk_df, A, X, Z = SymMNA.smna(net_list_cm)

The code below assembles the network equations from the MNA matrices and displays the equations.

In [10]:
# Put matrices into SymPy 
X = Matrix(X)
Z = Matrix(Z)

# formulate the symbolic network equations
NE_sym = Eq(A*X,Z)

# the free symbols are entered as SymPy variables and the element values are put into a dictionary.
var(str(NE_sym.free_symbols).replace('{','').replace('}',''))

# display the network equations
temp = ''
for i in range(shape(NE_sym.lhs)[0]):
    temp += '${:s} = {:s}$<br>'.format(latex(NE_sym.rhs[i]),latex(NE_sym.lhs[i]))
Markdown(temp)

$0 = I_{L1} + \frac{v_{1}}{R_{2}}$<br>$0 = C_{1} s v_{2} - I_{L1} + I_{L2}$<br>$0 = C_{3} s v_{3} - C_{3} s v_{6} - I_{L2} + I_{L4}$<br>$0 = - I_{L4} + I_{L5} + I_{V cm}$<br>$0 = C_{2} s v_{5} - C_{2} s v_{6} - I_{L3} - I_{L5}$<br>$0 = - C_{2} s v_{5} - C_{3} s v_{3} - I_{V cm} + v_{6} \left(C_{2} s + C_{3} s\right)$<br>$V_{cm} = v_{4} - v_{6}$<br>$0 = - I_{L1} L_{1} s + v_{1} - v_{2}$<br>$0 = - I_{L2} L_{2} s - I_{L3} M s + v_{2} - v_{3}$<br>$0 = - I_{L2} M s - I_{L3} L_{3} s - v_{5}$<br>$0 = - I_{L4} L_{4} s + v_{3} - v_{4}$<br>$0 = - I_{L5} L_{5} s + v_{4} - v_{5}$<br>

Solve the network equations for unknown currents and node voltages.

In [11]:
%%time
U_sym = solve(NE_sym,X)

CPU times: user 15.8 s, sys: 28 ms, total: 15.8 s
Wall time: 15.8 s


The transfer function for the common mode case is: $H(s)=\frac {v_2(s)}{v_4(s)}$. The code below takes the expressions for $v_2$ and $v_4$ to formulate the transfer function.

In [12]:
H_sym_cm = (U_sym[v2]/U_sym[v4]).simplify().collect(s)

Markdown('$H(s)={:s}$'.format(latex(H_sym_cm)))

$H(s)=\frac{C_{2} L_{5} R_{2} - C_{3} L_{4} R_{2} + s \left(C_{2} L_{1} L_{5} - C_{3} L_{1} L_{4}\right)}{C_{2} L_{5} R_{2} + s^{5} \left(C_{1} C_{2} C_{3} L_{1} L_{2} L_{4} L_{5} + C_{1} C_{2} C_{3} L_{1} L_{3} L_{4} L_{5} - 2 C_{1} C_{2} C_{3} L_{1} L_{4} L_{5} M\right) + s^{4} \left(C_{1} C_{2} C_{3} L_{2} L_{4} L_{5} R_{2} + C_{1} C_{2} C_{3} L_{3} L_{4} L_{5} R_{2} - 2 C_{1} C_{2} C_{3} L_{4} L_{5} M R_{2}\right) + s^{3} \left(C_{1} C_{2} L_{1} L_{2} L_{5} + C_{1} C_{2} L_{1} L_{4} L_{5} - C_{1} C_{2} L_{1} L_{5} M + C_{1} C_{3} L_{1} L_{3} L_{4} + C_{1} C_{3} L_{1} L_{4} L_{5} - C_{1} C_{3} L_{1} L_{4} M + C_{2} C_{3} L_{1} L_{4} L_{5} + C_{2} C_{3} L_{2} L_{4} L_{5} + C_{2} C_{3} L_{3} L_{4} L_{5} - 2 C_{2} C_{3} L_{4} L_{5} M\right) + s^{2} \left(C_{1} C_{2} L_{2} L_{5} R_{2} + C_{1} C_{2} L_{4} L_{5} R_{2} - C_{1} C_{2} L_{5} M R_{2} + C_{1} C_{3} L_{3} L_{4} R_{2} + C_{1} C_{3} L_{4} L_{5} R_{2} - C_{1} C_{3} L_{4} M R_{2} + C_{2} C_{3} L_{4} L_{5} R_{2}\right) + s \left(C_{2} L_{1} L_{5} + C_{2} L_{2} L_{5} + C_{2} L_{4} L_{5} - C_{2} L_{5} M + C_{3} L_{3} L_{4} + C_{3} L_{4} L_{5} - C_{3} L_{4} M\right)}$

The expression for the transfer function can be simplified by letting $L_2=L_3=L$, $C_2=C_3=C$, $L_5=L_4$ and $M=1$. 

In [13]:
H_sym_cm_simplified = H_sym_cm.subs({L2:L,L3:L,L5:L4,C2:C,C3:C,M:1}).simplify()

Markdown('$H(s)={:s}$'.format(latex(H_sym_cm_simplified)))

$H(s)=0$

The transfer function for the common mode case is zero after the substitutions are made. Hopefully, this answers your second question.

As you can see, the symbolic solutions to the network equations are often very long and usually don't offer too much insight into the function of the circuit. With the Python code used in my answer, the network equations are easy to generate from a circuit's netlist and the netlist is easy to export from most schematic capture tools.